In [1]:
import os
# 1. [关键] 必须设置缓存到数据盘 (50G硬盘保命设置)
os.environ["HF_HOME"] = "/root/autodl-tmp/hf_cache"
# 2. [关键] 关闭 HF_TRANSFER 加速 (解决 RuntimeError: no permits available)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# 3. [关键] 关闭 Xet 加速 (解决 CAS service error)
os.environ["HF_HUB_DISABLE_XET"] = "1"
# 4. [关键] 使用国内镜像 (解决连接超时)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("..")
from unsloth import FastLanguageModel, get_chat_template
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from src.utils.config_loader import load_config
import torch
import pandas as pd

cfg = load_config("../configs/config.yaml")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
student_model_id = cfg.model_student.model_id
max_seq_length = cfg.model_student.max_seq_length
load_in_4bit = cfg.model_student.load_in_4bit
dtype = cfg.model_student.dtype
trust_remote_code = cfg.model_student.trust_remote_code
print(student_model_id, max_seq_length, load_in_4bit, dtype, trust_remote_code)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=student_model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=trust_remote_code
)

unsloth/Qwen3-8B-unsloth-bnb-4bit 2048 True bfloat16 True
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
# 设置 PEFT (LoRA)
r = cfg.training.lora.r
target_modules = cfg.training.lora.target_modules
lora_alpha = cfg.training.lora.lora_alpha
lora_dropout = cfg.training.lora.lora_dropout
model = FastLanguageModel.get_peft_model(
    model,
    r=r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

model.print_trainable_parameters()

Unsloth 2025.11.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 174,587,904 || all params: 8,365,323,264 || trainable%: 2.0870


In [4]:
from unsloth.chat_templates import CHAT_TEMPLATES
print(list(CHAT_TEMPLATES.keys()))

['unsloth', 'zephyr', 'chatml', 'mistral', 'llama', 'vicuna', 'vicuna_old', 'vicuna old', 'alpaca', 'gemma', 'gemma_chatml', 'gemma2', 'gemma2_chatml', 'llama-3', 'llama3', 'phi-3', 'phi-35', 'phi-3.5', 'llama-3.1', 'llama-31', 'llama-3.2', 'llama-3.3', 'llama-32', 'llama-33', 'qwen-2.5', 'qwen-25', 'qwen25', 'qwen2.5', 'phi-4', 'gemma-3', 'gemma3', 'qwen-3', 'qwen3', 'gemma-3n', 'gemma3n', 'gpt-oss', 'gptoss', 'qwen3-instruct', 'qwen3-thinking', 'lfm-2', 'starling', 'yi-chat']


In [4]:
from src.templates import SYSTEM_PROMPT, USER_PROMPT

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-3"
)

def formatting_prompts_func(examples):
    inputs = examples["text"]
    aspects = examples["aspect"]
    outputs = examples["target_output"]

    texts = []
    for text, aspect, output in zip(inputs, aspects, outputs):
        conversation = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(aspect=aspect, text=text)},
            {"role": "assistant", "content": output}
        ]
        # 使用 apply_chat_template 转换为 token IDs 之前的文本格式
        # Unsloth 需要的是 'text' 字段包含格式化后的 Prompt
        formatted_prompt = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(formatted_prompt)
    
    return {"text": texts}

In [5]:
from datasets import Dataset
import re

# Unsloth 需要 specific 的格式，我们这里构造 ChatML 格式
rest_dataset = load_dataset("json", data_files="../data/processed/train_rest_cot_distilled.jsonl", split="train")
lap_dataset = load_dataset("json", data_files="../data/processed/train_lap_cot_distilled.jsonl", split="train")
print("餐厅数据集大小:", len(rest_dataset))
print("笔记本数据集大小:", len(lap_dataset))

from datasets import concatenate_datasets
train_dataset = concatenate_datasets([rest_dataset, lap_dataset])
print("训练集大小:", len(train_dataset))

# ==========================================
# 0. 辅助函数: CoT 质量检测
# ==========================================
def is_high_quality(row):
    """
    清洗规则：
    1. target_output 必须包含 <think> 标签
    2. 推理内容长度不能太短 (比如少于 10 个单词)
    3. 不能包含明确的失败关键词 (如 "Analysis failed")
    """
    content = row['target_output']
    
    # 提取 <think> 内部的内容
    match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    if not match:
        return False # 连标签都没有，丢弃
    
    rationale = match.group(1).strip()
    
    # 规则 A: 剔除失败的生成
    if "Analysis failed" in rationale or "analysis failed" in rationale:
        return False
        
    # 规则 B: 剔除推理太短的 (太短通常意味着没推理，或者只是重复了一遍)
    # 按空格分词，至少要有 15 个词 (根据你的数据情况微调)
    if len(rationale.split()) < 15:
        return False
        
    # 规则 C: 简单的重复检测 (防止复读机)
    # 如果同样的短语重复了太多次，可以视为低质量 (这里做个简单示例)
    if len(set(rationale.split())) < 5: # 词汇量极贫乏
        return False
        
    return True

# =================数据增强=================

# ==========================================
# 1. 将 Dataset 转换为 Pandas DataFrame 以便处理
# ==========================================
df_train = train_dataset.to_pandas()

# ==========================================
# 2. 过滤掉 Conflict 类别 (3-Way 策略)
# ==========================================
# 假设 polarity 存储在 'polarity' 字段中，如果是在 'target_output' 里，我们需要提取它
# 根据你的代码，原始数据列应该是 'polarity'
print(f"原始数据量: {len(df_train)}")

# 2.1 剔除 Conflict (基本操作)
# 过滤掉 conflict (注意：需确认你的数据中 conflict 的具体字符串)
df_train = df_train[df_train['polarity'] != 'conflict']
print(f"过滤 Conflict 后数据量: {len(df_train)}")

# 2.2 CoT 质量过滤 (核心清洗)
# 使用 apply 应用上面的 is_high_quality 函数
df_clean = df_train[df_train.apply(is_high_quality, axis=1)].copy()
dropped_count = len(df_train) - len(df_clean)
print(f"Step 2 - CoT 质量清洗后: {len(df_clean)} (丢弃了 {dropped_count} 条低质量/短推理数据)")

# ==========================================
# 3. 计算类别分布并执行“加权过采样”
# ==========================================
# 统计各类别数量
class_counts = df_clean['polarity'].value_counts()
print("\n当前类别分布:")
print(class_counts)

# 找到最大类别的数量 (通常是 Positive)
max_count = class_counts.max()

# [核心策略] 设定目标数量为最大类的 70%
# 如果完全平衡是 1.0，原始分布可能只有 0.2，我们取折中 0.7
# 这样既不会让模型忽视 Neutral，也不会让 Neutral 喧宾夺主导致误判 Negative
SOFT_RATIO = 0.7 
target_count = int(max_count * SOFT_RATIO)

print(f"目标采样数量 (Soft Target): {target_count} (Max Count 的 {SOFT_RATIO*100}%)")

balanced_dfs = []
for sentiment in class_counts.index:
    df_subset = df_clean[df_clean['polarity'] == sentiment]
    current_len = len(df_subset)
    
    # 只有当数量少于 target_count 时才过采样
    # 注意：Positive 肯定大于 target_count，所以 Positive 保持原样，不会被剪裁
    if current_len < target_count:
        print(f"  -> 类别 {sentiment} (数量 {current_len}) 过采样至 {target_count}")
        # replace=True 允许重复采样
        df_subset_upsampled = df_subset.sample(n=target_count, replace=True, random_state=42)
        balanced_dfs.append(df_subset_upsampled)
    else:
        print(f"  -> 类别 {sentiment} (数量 {current_len}) 保持不变")
        balanced_dfs.append(df_subset)

df_balanced = pd.concat(balanced_dfs)
# 打乱顺序
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nSoft Balancing 后类别分布:")
print(df_balanced['polarity'].value_counts())

# ==========================================
# 4. 转回 HuggingFace Dataset
# ==========================================
train_dataset = Dataset.from_pandas(df_balanced)

df_balanced.to_json("../data/processed/train_cot_cleaned_balanced_soft.jsonl", orient="records", lines=True, force_ascii=False)

餐厅数据集大小: 3693
笔记本数据集大小: 2358
训练集大小: 6051
原始数据量: 6051
过滤 Conflict 后数据量: 5915
Step 2 - CoT 质量清洗后: 5316 (丢弃了 599 条低质量/短推理数据)

当前类别分布:
polarity
positive    2884
negative    1464
neutral      968
Name: count, dtype: int64
目标采样数量 (Soft Target): 2018 (Max Count 的 70.0%)
  -> 类别 positive (数量 2884) 保持不变
  -> 类别 negative (数量 1464) 过采样至 2018
  -> 类别 neutral (数量 968) 过采样至 2018

Soft Balancing 后类别分布:
polarity
positive    2884
neutral     2018
negative    2018
Name: count, dtype: int64


In [6]:
# ==========================================
# 0. 定义提取和对比函数
# ==========================================
def check_sentiment_consistency(row):
    """
    检查 target_output 中的 Final Sentiment 是否与 polarity 一致
    返回: 'Match' (一致), 'Mismatch' (不一致), 'Parse Error' (解析失败)
    """
    ground_truth = row['polarity'].strip().lower()
    content = row['target_output']
    
    # 使用正则提取 Final Sentiment 后的单词
    # 兼容: "Final Sentiment: Positive", "Final Sentiment: positive.", "**Final Sentiment**: Positive"
    match = re.search(r"Final Sentiment:.*?([a-zA-Z]+)", content, re.IGNORECASE)
    
    if not match:
        return "Parse Error"
    
    # 提取出的预测情感
    predicted_sentiment = match.group(1).lower()
    
    # 简单的标准化 (防止 model 输出 "pos" 或 "positive." 等微小差异)
    if "pos" in predicted_sentiment: predicted_sentiment = "positive"
    if "neg" in predicted_sentiment: predicted_sentiment = "negative"
    if "neu" in predicted_sentiment: predicted_sentiment = "neutral"
    
    # 对比
    if predicted_sentiment == ground_truth:
        return "Match"
    else:
        # 这里把预测结果存下来，方便后续查看
        return f"Mismatch (GT:{ground_truth} vs Pred:{predicted_sentiment})"

# ==========================================
# 1. 执行检查
# ==========================================
print("正在检查教师模型的一致性...")
# 创建一个临时列来存储检查结果
consistency_results = df_clean.apply(check_sentiment_consistency, axis=1)

# ==========================================
# 2. 统计结果
# ==========================================
counts = consistency_results.value_counts()
total = len(df_clean)
match_count = counts.get("Match", 0)
parse_error_count = counts.get("Parse Error", 0)
mismatch_count = total - match_count - parse_error_count

print(f"\n📊 一致性检查报告 (Total: {total}):")
print(f"✅ 一致 (Match):        {match_count} ({match_count/total:.2%})")
print(f"❌ 不一致 (Mismatch):   {mismatch_count} ({mismatch_count/total:.2%})")
print(f"⚠️ 解析失败 (Error):    {parse_error_count} ({parse_error_count/total:.2%})")

# ==========================================
# 3. 查看“不一致”的样本 (这很重要！)
# ==========================================
# 找出不一致的索引
mismatch_indices = consistency_results[consistency_results.str.startswith("Mismatch")].index

if len(mismatch_indices) > 0:
    print(f"\n🔍 发现 {len(mismatch_indices)} 条错误样本，展示前 5 条:")
    print("-" * 60)
    for idx in mismatch_indices[:5]:
        row = df_clean.loc[idx]
        error_type = consistency_results.loc[idx]
        print(f"Index: {idx}")
        print(f"Text: {row['text']}")
        print(f"Aspect: {row.get('term', row.get('aspect', 'N/A'))}") # 兼容 term 或 aspect 字段名
        print(f"🔴 {error_type}") # 显示 GT vs Pred
        print(f"Rational: {row['target_output'].split('Final Sentiment')[0][-100:]}...") # 只看推理的最后一段
        print("-" * 60)
else:
    print("\n🎉 完美！没有发现不一致的数据。")

# ==========================================
# 4. (可选) 过滤策略建议
# ==========================================
# 如果你想直接把不一致的数据删掉，运行下面这行：
# df_final_clean = df_clean[consistency_results == "Match"].copy()
# print(f"\n🚀 过滤后最终用于训练的数据量: {len(df_final_clean)}")

正在检查教师模型的一致性...

📊 一致性检查报告 (Total: 5316):
✅ 一致 (Match):        5316 (100.00%)
❌ 不一致 (Mismatch):   0 (0.00%)
⚠️ 解析失败 (Error):    0 (0.00%)

🎉 完美！没有发现不一致的数据。


In [7]:
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/6920 [00:00<?, ? examples/s]

In [8]:
# 查看第一条数据经过格式化后的样子
print("=== 原始数据 (Input) ===")
print(f"Text: {train_dataset[0]['text']}")

print("\n=== 🔍 格式化后的完整 Prompt (送入模型的内容) ===")
print(train_dataset[0]["text"])

print("\n" + "="*50)
print("=== 检查要点 ===")
print("1. 开头是否有 <|im_start|>system ...")
print("2. 中间是否有 <|im_start|>user ...")
print("3. 最后是否有 <|im_start|>assistant ...")
print("4. 最重要的是：<think> 标签和推理过程是否完整都在 assistant 里？")

=== 原始数据 (Input) ===
Text: <|im_start|>system

You are a precise Aspect-Based Sentiment Analysis (ABSA) engine.
Your task is to analyze the review text provided by the user and follow these strict steps:

1. Aspect Identification: Focus strictly on the specific 'Aspect' provided in the user instruction.
2. Sentiment Judgment: Determine the sentiment polarity expressed by the reviewer towards this specific aspect.
3. Evidence Citation: Extract key phrases or adjectives from the original text that support your judgment.

You must use the <think> tags to execute this multi-step analysis process.
Inside the <think> tags, display your step-by-step reasoning, including the evidence you found.
After the <think> tags, provide the final conclusion.

Strict Output Format:
<think>
[Your detailed reasoning process steps...]
</think>
Final Sentiment: [Polarity]

The Polarity must be one of: Positive, Negative, Neutral, Conflict.
Do not include any additional explanatory text outside this format.
<|

In [9]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,  # 显式关闭 packing，对 CoT 逻辑学习更稳
    dataset_num_proc=2, # 加速 Tokenization 处理
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        
        num_train_epochs=3, # 按轮次训练 - 3轮
        warmup_steps=100,
        learning_rate=2e-4,
        lr_scheduler_type="cosine", # 多轮训练推荐用 cosine，后期学习率下降平滑，有助于收敛
        
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=42,
        output_dir="../outputs/qwen3_cot_finetuned_data_augmentation*0.7_LoraRank128",

        #每一个 epoch 保存一次模型，防止跑崩
        save_strategy="epoch",
        save_total_limit=3,      # 只保留最近的3个存档，防止硬盘爆满
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/6920 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,920 | Num Epochs = 3 | Total steps = 2,595
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 174,587,904 of 8,365,323,264 (2.09% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.842300
20,1.830300
30,0.949200
40,0.560100
50,0.485100
60,0.472100
70,0.472400
80,0.446700
90,0.425200
100,0.429300


In [14]:
model.save_pretrained("../outputs/qwen3_cot_finetuned_data_augmentation*0.7_LoraRank128")
tokenizer.save_pretrained("../outputs/qwen3_cot_finetuned_data_augmentation*0.7_LoraRank128")

('../outputs/qwen3_cot_finetuned_data_augmentation/tokenizer_config.json',
 '../outputs/qwen3_cot_finetuned_data_augmentation/special_tokens_map.json',
 '../outputs/qwen3_cot_finetuned_data_augmentation/chat_template.jinja',
 '../outputs/qwen3_cot_finetuned_data_augmentation/vocab.json',
 '../outputs/qwen3_cot_finetuned_data_augmentation/merges.txt',
 '../outputs/qwen3_cot_finetuned_data_augmentation/added_tokens.json',
 '../outputs/qwen3_cot_finetuned_data_augmentation/tokenizer.json')